# GeoLibre con código

**Curso avanzado de SIG en ecología · EBD-CSIC · septiembre de 2026**

[GeoLibre](https://opengeos.github.io/GeoLibre/) (Qiusheng Wu) es un SIG que funciona en el navegador. Además de la [versión web](https://web.geolibre.app), tiene un **paquete de Python** que mete **la aplicación completa dentro de una celda** del notebook. Así se puede combinar lo mejor de los dos mundos:

- **Con código**, cargamos datos, los filtramos con pandas y los llevamos al mapa. También podemos repetir el proceso las veces que haga falta.
- **Con la interfaz**, exploramos, cambiamos estilos, medimos y dibujamos.
- Las dos partes **se sincronizan**: lo que añades desde Python aparece en la interfaz, y lo que dibujas en la interfaz se puede leer desde Python.

| Parte | Qué hacemos |
|---|---|
| 1 | Primer mapa y mapas base |
| 2 | Capas vectoriales desde internet (GeoParquet, GeoJSON) |
| 3 | De pandas al mapa: filtrar y pintar los GPS de las aves |
| 4 | Mapa de calor y coropletas |
| 5 | Un ráster: el hidroperiodo de la marisma |
| 6 | Earth Engine y ndvi2gif dentro de GeoLibre, con comparador deslizante |
| 7 | Dibujar una zona en el mapa y analizarla con ndvi2gif |
| 8 | Guardar el proyecto y abrirlo en la web |

> Los datos son los del ejercicio **"La marisma en el tiempo"**. Los GPS son **simulados**; la inundación es **real** (Landsat, Protocolo v2).

## 0. Instalación

In [ ]:
%pip install -q geolibre

In [ ]:
import pandas as pd
import geopandas as gpd
from geolibre import Map

URL_DATOS = 'https://raw.githubusercontent.com/Digdgeo/sig-avanzado-2026/main/datos'

## 1. Primer mapa

`Map()` crea el mapa. Para que aparezca, la **última línea de la celda** tiene que ser el nombre del mapa (`m`).

Parámetros útiles:

| Parámetro | Qué hace | Valores |
|---|---|---|
| `center` | Centro del mapa | `(longitud, latitud)`. ¡Ojo, **primero la longitud**! |
| `zoom` | Nivel de zoom | de 0 a 22 |
| `basemap` | Mapa base | `'liberty'`, `'positron'`, `'dark'`, `'bright'`, `'fiord'` |
| `height` | Alto de la celda | `'600px'` |
| `layout` | Cuánta interfaz se ve | `'embed'`, `'full'` (con todos los menús), `'maponly'` |

In [ ]:
m = Map(center=(-6.37, 37.0), zoom=10, basemap='positron', height='650px')
m

Explora la interfaz: el panel de capas, los menús... Es la misma aplicación que la web.

Ahora cambiamos el mapa base **desde el código**. El mapa de arriba se actualiza solo, sin volver a crearlo. Añadimos también una **ortofoto** como capa de teselas (XYZ).

In [ ]:
m.add_basemap('dark')

In [ ]:
m.add_tile_layer(
    'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    name='Ortofoto (Esri)',
    attribution='Esri, Maxar, Earthstar Geographics',
)

## 2. Capas vectoriales desde internet

Hay dos caminos para traer capas vectoriales:

- `add_geojson(url)`: GeoLibre descarga el GeoJSON directamente.
- `add_gdf(gdf)`: lo leemos antes con **GeoPandas**, que abre casi cualquier formato (GeoParquet, GeoPackage, Shapefile...), y pasamos el GeoDataFrame.

> En la versión web, los GeoParquet se arrastran directamente al mapa. Desde el notebook, cargarlos por URL con `add_geoparquet` nos ha dado problemas, así que usamos GeoPandas.

Los estilos se pasan como argumentos: `fillColor`, `fillOpacity`, `strokeColor`, `strokeWidth`, `circleRadius`.

In [ ]:
recintos = gpd.read_file(f'{URL_DATOS}/ligero/recintos_marisma.geojson')

m2 = Map(center=(-6.37, 37.0), zoom=10, height='650px')
m2.add_geojson(
    f'{URL_DATOS}/ligero/inundacion_ligero.geojson',
    name='Inundación 2019-2020',
    fillColor='#2563eb', fillOpacity=0.3, strokeWidth=0,
)
m2.add_gdf(recintos, name='Recintos de la marisma',
           fillOpacity=0, strokeColor='#b91c1c', strokeWidth=2)
m2

Pincha en un polígono para ver sus atributos. Desde Python podemos preguntar qué capas tiene el mapa:

In [ ]:
m2.layer_names

## 3. De pandas al mapa

Aquí juntamos lo visto en la intro de Python: **leer una tabla, filtrarla y pintarla**.

In [ ]:
import urllib.request

urllib.request.urlretrieve(f'{URL_DATOS}/ligero/gps_aves_ligero.parquet', 'gps_aves_ligero.parquet')
gps = gpd.read_parquet('gps_aves_ligero.parquet').set_crs(4326, allow_override=True)

print(gps.shape)
gps[['id_ave', 'nombre_comun', 'timestamp', 'lon', 'lat']].head()

> **Importante:** GeoLibre guarda el mapa como **JSON**, y JSON no sabe qué es una fecha de pandas (`datetime64`). Además, todo lo que mandamos al mapa viaja entero al navegador. Por eso, antes de pintar, preparamos una copia **solo con las columnas que nos interesan** y **con las fechas como texto**. Para filtrar seguimos usando `gps`, que conserva las fechas de verdad.
>
> Es un buen momento para escribir una **función**, como en la intro de Python.

In [ ]:
COLUMNAS_MAPA = ['id_ave', 'nombre_comun', 'sexo', 'fecha_hora_local']

def para_mapa(gdf):
    copia = gdf[COLUMNAS_MAPA + [gdf.geometry.name]].copy()
    copia['fecha_hora_local'] = copia['fecha_hora_local'].dt.strftime('%Y-%m-%d %H:%M')
    return copia

gps['nombre_comun'].value_counts()

### Filtrar: flamencos en febrero de 2020

Las fechas se filtran con `.dt` (año, mes, día...). Las condiciones se combinan con `&` y cada una va entre paréntesis.

In [ ]:
flamencos_feb = gps[(gps['nombre_comun'] == 'Flamenco común') &
                    (gps['timestamp'].dt.year == 2020) &
                    (gps['timestamp'].dt.month == 2)]
print(len(flamencos_feb), 'posiciones')

m3 = Map(center=(-6.35, 37.0), zoom=10, basemap='positron', height='650px')
m3.add_gdf(para_mapa(flamencos_feb), name='Flamencos, febrero 2020', fillColor='#ec4899', strokeColor='#831843', strokeWidth=1, circleRadius=4)
m3

### Una capa por especie con un bucle `for`

Un **diccionario** asigna un color a cada especie. El bucle crea una capa por especie, siempre con el mismo código.

In [ ]:
colores = {
    'Flamenco común': '#ec4899',
    'Espátula común': '#f59e0b',
    'Ánsar común': '#10b981',
}

for especie, color in colores.items():
    datos = gps[gps['nombre_comun'] == especie]
    m3.add_gdf(para_mapa(datos), name=especie, fillColor=color, strokeColor=color, strokeWidth=0.5, circleRadius=3)

Vuelve al mapa de arriba: han aparecido tres capas nuevas, una por especie. Enciende y apaga capas desde el panel.

### ✏️ Prueba

1. Pinta solo las posiciones **nocturnas** de las espátulas. Pista: `gps['fecha_hora_local'].dt.hour`.
2. Pinta un único individuo (`id_ave`). ¿Cuántos individuos hay de cada especie?

## 4. Mapa de calor y coropletas

### Mapa de calor: ¿dónde se concentran?

In [ ]:
m4 = Map(center=(-6.35, 37.05), zoom=9.5, basemap='dark', height='650px')
m4.add_heatmap(para_mapa(gps[gps['nombre_comun'] == 'Flamenco común']), name='Densidad de flamencos',
               radius=10, intensity=0.3, color_ramp='turbo')
m4

### Coropletas: ¿qué recinto se inundó más?

La tabla `inundacion_por_recinto_2019_2020.csv` tiene el porcentaje inundado de cada recinto en cada fecha. Vamos a:

1. Calcular el **máximo** por recinto con `groupby`.
2. **Unirlo** a los polígonos con `merge`.
3. Pintarlo con `add_choropleth`: colores más oscuros para los recintos más inundados. Pincha en cada uno para ver su valor.

In [ ]:
inund = pd.read_csv(f'{URL_DATOS}/inundacion_por_recinto_2019_2020.csv')
resumen = inund.groupby('recinto')['pct_inundado'].agg(['max', 'mean']).round(1).reset_index()
resumen.columns = ['recinto', 'pct_inund_max', 'pct_inund_medio']
resumen

In [ ]:
recintos_inund = recintos.merge(resumen, on='recinto')

m4b = Map(center=(-6.35, 37.0), zoom=10, basemap='positron', height='650px')
m4b.add_choropleth(recintos_inund, column='pct_inund_max', name='% máximo inundado',
                   class_count=4, colormap='blues', fillOpacity=0.8)
m4b

## 5. Un ráster: el hidroperiodo

El **hidroperiodo** es el número de días que cada píxel estuvo inundado en el ciclo 2019-2020 (de 0 a 365). Es un GeoTIFF optimizado para la nube (**COG**): GeoLibre solo descarga los trozos que se ven en pantalla.

Con la paleta `viridis`, el morado oscuro es seco (0 días) y el amarillo es agua todo el año (el mar y las balsas).

In [ ]:
m5 = Map(center=(-6.35, 37.0), zoom=10, basemap='positron', height='650px')
m5.add_cog(
    'https://digdgeo.github.io/sig-avanzado-2026/datos/hidroperiodo_2019_2020_marisma.tif',
    name='Hidroperiodo 2019-2020 (días)',
    colormap='viridis',
    rescale=[[0, 250]],
)
m5

## 6. Earth Engine y ndvi2gif dentro de GeoLibre

`add_ee_layer()` pinta cualquier imagen de Earth Engine, igual que `geemap`. Así podemos **ver los composites de ndvi2gif en GeoLibre** y usar sus herramientas, como el **comparador deslizante** (`split_map`).

Comparamos el **NDVI máximo de invierno** (enero a marzo) de la marisma en dos años. ¿Cuál fue más húmedo?

In [ ]:
%pip install -q ndvi2gif

In [ ]:
import ee
from ndvi2gif import NdviSeasonality

PROYECTO = 'ee-tunombre'   # <-- ¡cámbialo!
ee.Authenticate()
ee.Initialize(project=PROYECTO)

In [ ]:
marisma = ee.Geometry.Rectangle([-6.50, 36.84, -6.24, 37.16])

donana = NdviSeasonality(roi=marisma, sat='S2', index='ndvi', periods=4, key='max',
                         start_year=2021, end_year=2023)

# get_period_composite(año, periodo): periodo 0 = invierno
invierno_2021 = donana.get_period_composite(2021, 0).clip(marisma)
invierno_2023 = donana.get_period_composite(2023, 0).clip(marisma)

vis = {'min': 0, 'max': 0.8, 'palette': ['a0522d', 'f5deb3', 'ffffbf', '7fbf7b', '1b7837']}

m6 = Map(center=(-6.37, 37.0), zoom=10, height='650px')
capa_2021 = m6.add_ee_layer(invierno_2021, vis, name='NDVI invierno 2021')
capa_2023 = m6.add_ee_layer(invierno_2023, vis, name='NDVI invierno 2023')
m6.split_map(left_layers=[capa_2021], right_layers=[capa_2023])
m6

Arrastra la barra central: a la izquierda está 2021 y a la derecha 2023. Los colores van de marrón (NDVI bajo) a verde oscuro (NDVI alto).

### ✏️ Prueba

Cambia `index='ndvi'` por `index='mndwi'` y la paleta por azules (`'palette': ['ffffff', '2563eb']`, `'min': -0.3`, `'max': 0.5`). ¿Qué año tuvo más agua en invierno?

## 7. Dibujar una zona y analizarla con ndvi2gif

Aquí la interfaz y el código **trabajan juntos**:

1. **En el mapa**, abre el **editor geográfico** (en el panel *Capas*, el tercer icono de la barra superior: un lápiz sobre un cuadrado) y dibuja un **polígono** pequeño: un recinto, una laguna, un trozo de pinar...
2. **En la celda siguiente**, Python lee lo que has dibujado y ndvi2gif calcula su serie de NDVI.

In [ ]:
m7 = Map(center=(-6.37, 37.0), zoom=11, height='650px', layout='full')
m7.add_tile_layer(
    'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    name='Ortofoto (Esri)',
)
m7

In [ ]:
dibujo = m7.get_drawn_features()
print(len(dibujo), 'geometría(s) dibujada(s)')

if len(dibujo) > 0:
    zona = ee.Geometry(gpd.GeoDataFrame.from_features(dibujo, crs=4326).union_all().__geo_interface__)
else:
    print('No has dibujado nada: usamos una zona de arrozal de ejemplo')
    zona = ee.Geometry.Rectangle([-6.21, 37.13, -6.18, 37.16])

print(f'Superficie: {zona.area(1).divide(1e4).getInfo():.1f} ha')

In [ ]:
import matplotlib.pyplot as plt

mi_zona = NdviSeasonality(roi=zona, sat='S2', index='ndvi', periods=12, key='median',
                          start_year=2019, end_year=2024)

# Todas las medianas mensuales en el servidor y un único getInfo()
imagenes = []
for anio in range(2019, 2025):
    for mes in range(12):
        img = mi_zona.get_period_composite(anio, mes)
        imagenes.append(img.set('fecha', f'{anio}-{mes + 1:02d}-15'))

def media_zona(img):
    valor = img.reduceRegion(ee.Reducer.mean(), zona, 10, maxPixels=1e9, bestEffort=True)
    # Si en un mes no hay imágenes, el composite no tiene bandas: ponemos -999 y luego lo quitamos
    return ee.Feature(None, {'fecha': img.get('fecha'), 'ndvi': valor.get('nd', -999)})

res = ee.FeatureCollection(ee.ImageCollection(imagenes).map(media_zona)).getInfo()
serie = pd.DataFrame([f['properties'] for f in res['features']])
serie['fecha'] = pd.to_datetime(serie['fecha'])
serie = serie[serie['ndvi'] > -1]

serie.plot(x='fecha', y='ndvi', marker='.', figsize=(11, 4), legend=False, title='NDVI mensual de mi zona')
plt.grid(alpha=0.3)

Dibuja **otra zona distinta** (por ejemplo, arrozal frente a marisma natural), vuelve a ejecutar las dos celdas anteriores y compara los calendarios.

## 8. Guardar el proyecto y abrirlo en la web

Todo el estado del mapa (capas, estilos, vista) es un **proyecto** en formato JSON. Desde Python podemos:

- Leer el estado, **incluidos los cambios hechos a mano en la interfaz**.
- Guardarlo como `.geolibre.json`.
- Exportarlo como una **página HTML** independiente para compartir.

In [ ]:
m3.describe()

In [ ]:
m3.save_project('aves_marisma.geolibre.json')
m3.to_html('aves_marisma.html', title='Aves de la marisma')

try:
    from google.colab import files
    files.download('aves_marisma.geolibre.json')
    files.download('aves_marisma.html')
except ImportError:
    print('Guardados en la carpeta de trabajo')

El `.geolibre.json` se puede abrir en [web.geolibre.app](https://web.geolibre.app) desde el menú *Proyecto*. Así se puede **preparar un mapa con código y terminarlo a mano en la web**, o al revés.

> Las capas de Earth Engine dependen de un identificador temporal: en un proyecto guardado pueden dejar de verse al cabo de unas horas.

---

## Resumen

| Quiero... | Uso |
|---|---|
| Crear un mapa | `m = Map(center=(lon, lat), zoom=..., basemap=...)` |
| Cambiar el mapa base | `m.add_basemap('dark')` |
| GeoJSON desde URL | `add_geojson(url)` |
| Capa desde un GeoDataFrame | `add_gdf(gdf, name=..., fillColor=...)` |
| Mapa de calor | `add_heatmap(gdf_puntos)` |
| Coropletas | `add_choropleth(gdf, column=...)` |
| Ráster COG | `add_cog(url, colormap=..., rescale=...)` |
| Ortofoto o teselas XYZ | `add_tile_layer(url)` |
| Imagen de Earth Engine | `add_ee_layer(imagen, vis, name=...)` |
| Comparador deslizante | `split_map(left_layers=..., right_layers=...)` |
| Leer lo dibujado | `get_drawn_features(as_gdf=True)` |
| Guardar | `save_project(...)`, `to_html(...)` |

Documentación completa: [opengeos.github.io/GeoLibre](https://opengeos.github.io/GeoLibre/) · Ejemplos: [github.com/opengeos/GeoLibre](https://github.com/opengeos/GeoLibre/tree/main/python/examples)